In [1]:
import pandas as pd
import numpy as np
import os
import math

# Configuration
TARGET_STEPS = 5000 
DOWNSAMPLE_FACTOR = 4  # Assumes source is 200Hz -> Targets 50Hz (0.02s)

# DRT_RL Files to process
file_paths = [
    "/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/high_delay.xlsx",
    "/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/high_variance.xlsx",
    "/media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/low_delay.xlsx"
]

# Standardize Header Mapping
# Maps 'leader_ee_pose_point.x' -> 'leader_ee_pos_x'
COLUMN_MAPPING = {
    'leader_ee_pose_point.x': 'leader_ee_pos_x',
    'leader_ee_pose_point.y': 'leader_ee_pos_y',
    'leader_ee_pose_point.z': 'leader_ee_pos_z',
    'follower_ee_pose_point.x': 'follower_ee_pos_x',
    'follower_ee_pose_point.y': 'follower_ee_pos_y',
    'follower_ee_pose_point.z': 'follower_ee_pos_z'
}

def augment_data_pattern_repeat(df, target_length):
    """
    Extends the dataframe to `target_length` by cyclically repeating the 
    existing trajectory pattern.
    """
    current_length = len(df)
    
    if current_length >= target_length:
        return df.head(target_length)
    
    # Calculate repetition factor
    repeat_factor = math.ceil(target_length / current_length)
    print(f"    -> Augmenting: Pattern length {current_length} repeated {repeat_factor} times...")
    
    # Tile the dataframe
    df_repeated = pd.concat([df] * repeat_factor, ignore_index=True)
    
    # Truncate to exact target length
    return df_repeated.head(target_length)

def process_drt_rl_data(file_path):
    if not os.path.exists(file_path):
        print(f"Error: File not found at {file_path}")
        return

    try:
        print(f"Processing {os.path.basename(file_path)}...")
        
        # 1. Load Excel
        df = pd.read_excel(file_path, engine='openpyxl')
        
        # 2. Rename Columns
        # Only rename if the old columns exist, to avoid errors if already cleaned
        df.rename(columns=COLUMN_MAPPING, inplace=True)
        
        # Validation
        required_cols = ['leader_ee_pos_x', 'follower_ee_pos_x']
        if not all(col in df.columns for col in required_cols):
            print(f"  - Warning: Standard columns not found. Available: {df.columns.tolist()}")
            # Attempt to proceed anyway if data happens to be in correct columns but unchecked
            return

        # 3. Downsample (Temporal scaling)
        # We enforce the factor 4 (200Hz -> 50Hz)
        df_downsampled = df.iloc[::DOWNSAMPLE_FACTOR].reset_index(drop=True)
        print(f"  - Downsampled shape: {df_downsampled.shape}")

        # 4. Augment Pattern (Ensure 5000 steps)
        df_final = augment_data_pattern_repeat(df_downsampled, TARGET_STEPS)
        print(f"  - Final shape: {df_final.shape}")

        # 5. Save as CSV
        file_dir, file_name = os.path.split(file_path)
        name_root, _ = os.path.splitext(file_name)
        output_path = os.path.join(file_dir, f"{name_root}_cleaned.csv")
        
        df_final.to_csv(output_path, index=False)
        print(f"  - Saved to: {output_path}\n")

    except Exception as e:
        print(f"An error occurred: {e}\n")

if __name__ == "__main__":
    print("Starting DRT_RL Data Processing...\n")
    for path in file_paths:
        process_drt_rl_data(path)
    print("Processing Complete.")

Starting DRT_RL Data Processing...

Processing high_delay.xlsx...
  - Downsampled shape: (603, 10)
    -> Augmenting: Pattern length 603 repeated 9 times...
  - Final shape: (5000, 10)
  - Saved to: /media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/high_delay_cleaned.csv

Processing high_variance.xlsx...
  - Downsampled shape: (602, 10)
    -> Augmenting: Pattern length 602 repeated 9 times...
  - Final shape: (5000, 10)
  - Saved to: /media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/high_variance_cleaned.csv

Processing low_delay.xlsx...
  - Downsampled shape: (602, 10)
    -> Augmenting: Pattern length 602 repeated 9 times...
  - Final shape: (5000, 10)
  - Saved to: /media/kai/NewDisk/Kai_thesis/Master_Thesis_E2E_RL_Teleop/libfranka_ws/src/E2E_Teleoperation/E2E_Teleoperation/evaluation/DRT_RL/low_delay_cleaned.csv

Processing C